[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-05-04-numpy-broadcasting/challenge.ipynb)

# Broadcasting Without Loops — The Shape Rules That Make NumPy Fast

**Day 16 | ML Engineering | Monday short-coding**

**Relevance:** NumPy broadcasting is the floor beneath every mech interp implementation you'll write — pairwise distances appear in probing, masked attention in circuit analysis, row normalization in feature geometry. LASR CodeSignal Module 3 (Python ML coding) expects fluent vectorized NumPy; loop-free implementations are a direct signal of ML engineering proficiency.

**Goal:** implement three functions — pairwise L2 distances, row standardization, and causal attention masking — using broadcasting, no Python loops.

## Setup

In [ ]:
import numpy as np
np.random.seed(42)

## Background

NumPy's broadcasting rules let you write operations that automatically expand arrays to compatible shapes — no Python `for` loops needed.

**The rules, applied right-to-left across dimensions:**
- Two dimensions are compatible if they are **equal**, or **one of them is 1**.
- A size-1 dimension is virtually repeated to match the other.

```
(n, 1, d)  broadcasting with  (1, n, d)  →  (n, n, d)
(n, 1)     broadcasting with  (n, d)     →  (n, d)
```

Mastering this lets you express batch operations in one line that would otherwise require nested loops — and run at C speed instead of Python speed.

The three tasks below are real patterns from production ML and mech interp code.

## Task 1 — Pairwise L2 Distances

**`pairwise_l2(X)`** — given `X` of shape `(n, d)`, return a distance matrix `D` of shape `(n, n)` where:
```
D[i, j] = ||X[i] - X[j]||_2
```
No loops. No `scipy.spatial.distance`. Pure broadcasting + NumPy.

**Why this matters:** k-NN probing (checking whether a linear probe's neighbours cluster by label) uses exactly this. Efficient pairwise distance is also the core of t-SNE and UMAP.

In [ ]:
def pairwise_l2(X: np.ndarray) -> np.ndarray:
    """
    Compute all-pairs L2 distances.
    Args:
        X: shape (n, d)
    Returns:
        D: shape (n, n), D[i,j] = ||X[i]-X[j]||_2
    """
    raise NotImplementedError

### Tests — Task 1

In [ ]:
np.random.seed(42)
_X1 = np.random.randn(5, 3)
_D = pairwise_l2(_X1)

assert _D.shape == (5, 5), f"Expected (5,5), got {_D.shape}"
assert np.allclose(np.diag(_D), 0), "Self-distances should be 0"
assert np.allclose(_D, _D.T, atol=1e-10), "Distance matrix must be symmetric"

# Spot-check two pairs against a manual calculation
def _ref_l2(a, b): return float(np.sqrt(((a - b) ** 2).sum()))
for _i, _j in [(0, 2), (1, 4), (3, 0)]:
    assert np.isclose(_D[_i, _j], _ref_l2(_X1[_i], _X1[_j])), \
        f"Mismatch at ({_i},{_j}): got {_D[_i,_j]:.4f}, expected {_ref_l2(_X1[_i], _X1[_j]):.4f}"

print("Task 1 ✓")

## Task 2 — Row Standardization

**`standardize_rows(X, eps=1e-8)`** — given `X` of shape `(n, d)`, return a matrix where **each row** has zero mean and unit standard deviation.

No loops. Use `axis` and `keepdims` parameters strategically.

**Why this matters:** Standardizing activations row-by-row before a linear probe prevents high-variance features from dominating. The same op appears in layer norm (normalize over the *feature* axis, not the batch axis).

In [ ]:
def standardize_rows(X: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Normalize each row to zero mean and unit standard deviation.
    Args:
        X:   shape (n, d)
        eps: small constant for numerical stability
    Returns:
        X_std: shape (n, d)
    """
    raise NotImplementedError

### Tests — Task 2

In [ ]:
np.random.seed(99)
_X2 = np.random.randn(4, 6) * 3 + 5   # non-zero mean, non-unit std
_Xs = standardize_rows(_X2)

assert _Xs.shape == (4, 6), f"Shape mismatch: {_Xs.shape}"
assert np.allclose(_Xs.mean(axis=1), 0, atol=1e-6), \
    f"Row means should be ~0, got {_Xs.mean(axis=1)}"
assert np.allclose(_Xs.std(axis=1), 1, atol=1e-6), \
    f"Row stds should be ~1, got {_Xs.std(axis=1)}"

# Edge case: single-element row (all same value) shouldn't crash
_X_const = np.ones((2, 4))
_Xs_const = standardize_rows(_X_const)
assert not np.any(np.isnan(_Xs_const)), "No NaNs on constant row (eps guards this)"

print("Task 2 ✓")

## Task 3 — Causal Attention Mask

**`causal_mask_logits(logits)`** — given `logits` of shape `(T, T)`, return a copy where every position where `j > i` (a future token's key attending from a past query) is set to `-np.inf`. The lower triangle (including the diagonal) must be unchanged.

No `np.tril`/`np.triu` calls that hardcode the condition. Use index broadcasting to construct the boolean mask explicitly — the same technique you'll use when you need asymmetric or sliding-window masks.

**Why this matters:** Causal masking is the mechanism that prevents a language model from 'cheating' by attending to future tokens. Building it via index broadcasting generalises: the same pattern handles sliding-window attention, cross-attention masks, and position-dependent ignore patterns.

In [ ]:
def causal_mask_logits(logits: np.ndarray) -> np.ndarray:
    """
    Apply a causal mask: set logits[i, j] = -inf wherever j > i.
    Args:
        logits: shape (T, T)
    Returns:
        masked: shape (T, T), same as logits but upper-triangle = -inf
    """
    raise NotImplementedError

### Tests — Task 3

In [ ]:
np.random.seed(7)
_L = np.random.randn(4, 4)
_Lm = causal_mask_logits(_L)

assert _Lm.shape == (4, 4), f"Shape mismatch: {_Lm.shape}"

# Lower triangle (including diagonal) must be unchanged
for _i in range(4):
    for _j in range(_i + 1):   # j <= i
        assert _Lm[_i, _j] == _L[_i, _j], \
            f"Lower triangle changed at ({_i},{_j}): {_Lm[_i,_j]} != {_L[_i,_j]}"

# Upper triangle must be -inf
for _i in range(4):
    for _j in range(_i + 1, 4):   # j > i
        assert _Lm[_i, _j] == -np.inf, \
            f"Expected -inf at ({_i},{_j}), got {_Lm[_i,_j]}"

# Input should not be mutated
_L_orig = _L.copy()
_ = causal_mask_logits(_L)
assert np.allclose(_L, _L_orig), "Function must not mutate the input array"

print("Task 3 ✓")

## Reflection

All three tasks use the same underlying insight: insert size-1 dimensions (`None` / `np.newaxis`) to force NumPy to compare every pair of elements simultaneously — turning an O(n²) Python loop into a single vectorised C operation. This is the shape algebra that makes transformer forward passes tractable at scale.

<details>
<summary><b>Solution</b></summary>

```python
def pairwise_l2(X):
    diff = X[:, None, :] - X[None, :, :]  # (n, 1, d) - (1, n, d) → (n, n, d)
    return np.sqrt((diff ** 2).sum(axis=-1))

def standardize_rows(X, eps=1e-8):
    mu  = X.mean(axis=1, keepdims=True)   # (n, 1) — keepdims is essential
    std = X.std(axis=1, keepdims=True)     # (n, 1)
    return (X - mu) / (std + eps)

def causal_mask_logits(logits):
    T = logits.shape[0]
    i = np.arange(T)[:, None]  # column vector (T, 1)
    j = np.arange(T)[None, :]  # row vector    (1, T)
    future_mask = j > i        # (T, T) — True where position j is in the future
    out = logits.copy()
    out[future_mask] = -np.inf
    return out
```

**Key insight — Task 1:** `X[:, None, :]` has shape `(n, 1, d)` and `X[None, :, :]` has shape `(1, n, d)`. Broadcasting expands both to `(n, n, d)` — every pair of rows is differenced simultaneously, giving you the full O(n²) distance matrix with no Python loop.

**Key insight — Task 2:** Without `keepdims=True`, `X.mean(axis=1)` returns shape `(n,)` — a 1D array that does NOT broadcast cleanly against `(n, d)`. With `keepdims=True`, you get `(n, 1)`, which broadcasts correctly against every column. This `keepdims` habit is worth drilling until it's automatic.

**Key insight — Task 3:** `np.arange(T)[:, None]` (shape `(T,1)`) and `np.arange(T)[None, :]` (shape `(1,T)`) compare every (i, j) pair via broadcasting — giving a `(T, T)` boolean matrix without a double loop. This index-comparison pattern is what transformer libraries use internally for arbitrary attention patterns (sliding window, prefix masking, cross-attention), so recognizing it is worth more than knowing `np.tril`.
</details>